In [1]:
# Install Required Libraries

!pip install sentence-transformers nltk newspaper3k lxml_html_clean -q

In [2]:
#  Import Libraries

import nltk
from nltk.tokenize import sent_tokenize
from newspaper import Article
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

nltk.download('punkt')
nltk.download('punkt_tab')

print("All libraries imported successfully!")

All libraries imported successfully!


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mafai\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mafai\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [3]:
# STEP 1 - Read the File & Load First 700 Characters

url = (
    "https://www.washingtonpost.com/world/2025/06/13/"
    "air-india-plane-crash-survivor-vishwash-kumar-ramesh/"
)

article = Article(url)
article.download()
article.parse()

full_text = article.text

# Load first 700 characters
text_700 = full_text[:700]


print("STEP 1: First 700 Characters of the Article")

print(text_700)
print(f"\n Total characters loaded: {len(text_700)}")

STEP 1: First 700 Characters of the Article
Only one person on board Air India Flight 171 survived — British national Viswashkumar Ramesh, who could be seen limping past a crowd of shocked rescuers toward an ambulance shortly after the crash killed the other 241 passengers and crew members, as well as dozens of people on the ground in Ahmedabad.

Ramesh, 40, has been described as the “miracle in seat 11A” in British media, and several top Indian officials — including Prime Minister Narendra Modi — have visited him in the hospital.

“I don’t know how I survived,” Ramesh said in an interview from his hospital bed with broadcaster Doordarshan on Friday, with one arm heavily bandaged and a bloodied cut under his eye.

“I was on the side o

 Total characters loaded: 700


In [13]:
#  STEP 2 - Split Text into Sentences using NLTK

sentences = sent_tokenize(text_700)


print("STEP 2: Sentences Extracted via NLTK Tokenizer")


for i, sent in enumerate(sentences):
    print(f"\n[Sentence {i+1}]: {sent}")

print(f"\n✅ Total sentences found: {len(sentences)}")

STEP 2: Sentences Extracted via NLTK Tokenizer

[Sentence 1]: Only one person on board Air India Flight 171 survived — British national Viswashkumar Ramesh, who could be seen limping past a crowd of shocked rescuers toward an ambulance shortly after the crash killed the other 241 passengers and crew members, as well as dozens of people on the ground in Ahmedabad.

[Sentence 2]: Ramesh, 40, has been described as the “miracle in seat 11A” in British media, and several top Indian officials — including Prime Minister Narendra Modi — have visited him in the hospital.

[Sentence 3]: “I don’t know how I survived,” Ramesh said in an interview from his hospital bed with broadcaster Doordarshan on Friday, with one arm heavily bandaged and a bloodied cut under his eye.

[Sentence 4]: “I was on the side o

✅ Total sentences found: 4


In [14]:
# Cell 5: STEP 3A - Load Pre-trained Sentence Embedding Model

print("STEP 3A: Loading Pre-trained Sentence Embedding Model")


model = SentenceTransformer('all-MiniLM-L6-v2')

print("✅ Model Loaded: all-MiniLM-L6-v2")
print("   - Lightweight & fast transformer model")
print("   - Produces 384-dimensional sentence embeddings")
print("   - Trained on large-scale sentence similarity datasets")

STEP 3A: Loading Pre-trained Sentence Embedding Model


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Model Loaded: all-MiniLM-L6-v2
   - Lightweight & fast transformer model
   - Produces 384-dimensional sentence embeddings
   - Trained on large-scale sentence similarity datasets


In [15]:
#  STEP 3B - TF-IDF Vectorization on First 10 Sentences

print("STEP 3B: TF-IDF Vectorization on First 10 Sentences")


first_10 = sentences[:10] if len(sentences) >= 10 else sentences
print(f"Number of sentences being vectorized: {len(first_10)}")

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(first_10)

print(f"\nTF-IDF Matrix Shape         : {tfidf_matrix.shape}")
print(f"  → Rows                    : {tfidf_matrix.shape[0]} (sentences)")
print(f"  → Columns                 : {tfidf_matrix.shape[1]} (unique tokens)")
print(f"\nVocabulary Size             : {len(tfidf_vectorizer.vocabulary_)}")
print(f"\nFirst 20 TF-IDF Features    :")
print(list(tfidf_vectorizer.get_feature_names_out()[:20]))

print("\nTF-IDF Matrix (Dense) - First 3 Sentences x First 5 Tokens:")
feature_names = tfidf_vectorizer.get_feature_names_out()
dense = np.round(tfidf_matrix.toarray(), 3)
for i in range(min(3, len(first_10))):
    print(f"  Sentence {i+1}: {dense[i][:5]}")

STEP 3B: TF-IDF Vectorization on First 10 Sentences
Number of sentences being vectorized: 4

TF-IDF Matrix Shape         : (4, 87)
  → Rows                    : 4 (sentences)
  → Columns                 : 87 (unique tokens)

Vocabulary Size             : 87

First 20 TF-IDF Features    :
['11a', '171', '241', '40', 'after', 'ahmedabad', 'air', 'ambulance', 'an', 'and', 'arm', 'as', 'bandaged', 'be', 'bed', 'been', 'bloodied', 'board', 'british', 'broadcaster']

TF-IDF Matrix (Dense) - First 3 Sentences x First 5 Tokens:
  Sentence 1: [0.    0.142 0.142 0.    0.142]
  Sentence 2: [0.189 0.    0.    0.189 0.   ]
  Sentence 3: [0. 0. 0. 0. 0.]


In [16]:
#  STEP 4 - Generate Sentence Embeddings

print("STEP 4: Generating Embeddings for Each Sentence")

embeddings = model.encode(sentences)

print(f"Total Sentences Embedded          : {len(sentences)}")
print(f"Embedding Dimension per Sentence  : {embeddings[0].shape[0]}")
print(f"\n Shape of Embedding (Sentence 1) : {embeddings[0].shape}")
print(f"\nFirst 10 values of Embedding [Sentence 1]:")
print(np.round(embeddings[0][:10], 5))

STEP 4: Generating Embeddings for Each Sentence
Total Sentences Embedded          : 4
Embedding Dimension per Sentence  : 384

 Shape of Embedding (Sentence 1) : (384,)

First 10 values of Embedding [Sentence 1]:
[ 0.06274 -0.04708 -0.0324  -0.0177   0.07797  0.00929  0.06419  0.05162
 -0.05488  0.03848]


In [17]:
# Cell 8: STEP 5 - Compute Cosine Similarity

print("STEP 5: Cosine Similarity Between Sentence 1 & Sentence 2")


if len(embeddings) >= 2:
    sim_score = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]

    print(f"\n Sentence 1:\n   {sentences[0]}")
    print(f"\n Sentence 2:\n   {sentences[1]}")
    print(f"\n Cosine Similarity Score  : {sim_score:.4f}")

    # Interpretation
    if sim_score > 0.7:
        interpretation = " High Similarity    — Sentences are very closely related."
    elif sim_score > 0.4:
        interpretation = " Moderate Similarity — Sentences share some context."
    else:
        interpretation = " Low Similarity     — Sentences are quite different."

    print(f"   Interpretation          : {interpretation}")

else:
    print(" Not enough sentences to compute similarity.")
    print("   The 700-character limit may have produced only 1 sentence.")

STEP 5: Cosine Similarity Between Sentence 1 & Sentence 2

 Sentence 1:
   Only one person on board Air India Flight 171 survived — British national Viswashkumar Ramesh, who could be seen limping past a crowd of shocked rescuers toward an ambulance shortly after the crash killed the other 241 passengers and crew members, as well as dozens of people on the ground in Ahmedabad.

 Sentence 2:
   Ramesh, 40, has been described as the “miracle in seat 11A” in British media, and several top Indian officials — including Prime Minister Narendra Modi — have visited him in the hospital.

 Cosine Similarity Score  : 0.4220
   Interpretation          :  Moderate Similarity — Sentences share some context.
